# <center> <img src="figs/LogoUFSCar.jpg" alt="Logo UFScar" width="110" align="left"/>  <br/> <center>Universidade Federal de São Carlos (UFSCar)<br/><font size="4"> Departamento de Computação, campus Sorocaba</center></font>
</p>

<font size="4"><center><b>Disciplina: Processamento de Linguagem Natural</b></center></font>
  
<font size="3"><center>Prof. Dr. Tiago A. Almeida</center></font>

## <center>Projeto Final</center>

**Nome**: Anne Mari Suenaga Sakai   -   **RA**: 822304

**Nome**: Felipe Jun Nishitani   -   **RA**: 

**Nome**: Lucas Pereira Goes   -   **RA**: 

---
### Análise exploratória

Nesta seção, deve ser feita a leitura da base de dados e todas as análises necessárias para interpretar e analisar os dados, tais como:
* Significado de cada atributo
* Medidas descritivas
* Gráficos

In [432]:
# ============================================================
# 1. Carregamento e configuração inicial
# ============================================================

import pandas as pd
import numpy as np
import re

import os
os.environ["MPLBACKEND"] = "agg"

import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Ajuste o caminho se necessário
CAMINHO_TREINO = "train.csv"

df = pd.read_csv(CAMINHO_TREINO)

df.head()

,Id,Body,Category
0,12980,"{""conclusÃ£o diante exposto recorrente espera ...",3
1,9775,"{""ARTIGO_55 gecen a gacen serÃ£o devidas aos t...",3
2,16061,"{""seguir desacerto decisÃ£o que negou seguimen...",1
3,22984,"{""advocacia geral uniÃ£o procuradoria uniÃ£o p...",3
4,5716,"{""ministÃ©rio fazenda procuradoria geral fazen...",3


In [433]:
# ============================================================
# 1.1 Informações gerais da base
# ============================================================

print("Formato da base:", df.shape)
print("\nColunas:")
print(df.columns.tolist())

print("\nInformações gerais:")
display(df.info())

print("\nAmostras:")
display(df.sample(5, random_state=42))

Formato da base: (22680, 3)

Colunas:
['Id', 'Body', 'Category']

Informações gerais:
<class 'pandas.DataFrame'>
RangeIndex: 22680 entries, 0 to 22679
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Id        22680 non-null  int64
 1   Body      22680 non-null  str  
 2   Category  22680 non-null  int64
dtypes: int64(2), str(1)
memory usage: 531.7 KB


None


Amostras:


,Id,Body,Category
1087,23423,"{""advocacia geral uniÃ£o procuradoria geral fe...",-1
15256,21007,"{""advocacia geral uniÃ£o procuradoria uniÃ£o p...",-1
4679,6611,"{""servidores autoras sÃ£o pensionistas jÃ¡ rec...",-1
14368,1732,"{""estado minas gerais advocacia geral estado s...",-1
15241,835,"{""efigÃªnia camilo silva advocacia consultoria...",-1


In [434]:
# ============================================================
# 1.2 Significado dos atributos
# ============================================================

descricao_atributos = pd.DataFrame({
    "Atributo": ["Id", "Body", "Category"],
    "Significado": [
        "Identificador único do documento.",
        "Texto extraído do documento jurídico.",
        "Classe do documento jurídico."
    ]
})

display(descricao_atributos)

,Atributo,Significado
0,Id,Identificador único do documento.
1,Body,Texto extraído do documento jurídico.
2,Category,Classe do documento jurídico.


In [435]:
# ============================================================
# 1.3 Mapeamento das classes
# ============================================================

mapa_classes = {
    0: "Acórdão",
    1: "Agravo de Recurso Extraordinário (ARE)",
    2: "Despacho",
    3: "Recurso Extraordinário (RE)",
    4: "Sentença"
}

df["Category_nome"] = df["Category"].map(mapa_classes)

display(df[["Category", "Category_nome"]].drop_duplicates().sort_values("Category"))

,Category,Category_nome
34,-1,NaN
14,0,Acórdão
2,1,Agravo de Recurso Extraordinário (ARE)
8,2,Despacho
0,3,Recurso Extraordinário (RE)
12,4,Sentença


In [436]:
# ============================================================
# 1.4 Verificação de valores ausentes e duplicados
# ============================================================

print("Valores nulos por coluna:")
display(df.isnull().sum())

print("\nQuantidade de documentos duplicados por Id:")
print(df["Id"].duplicated().sum())

print("\nQuantidade de textos duplicados:")
print(df["Body"].duplicated().sum())

Valores nulos por coluna:


Id                  0
Body                0
Category            0
Category_nome    2268
dtype: int64


Quantidade de documentos duplicados por Id:
0

Quantidade de textos duplicados:
11183


In [437]:
# ============================================================
# 2. Distribuição das classes
# ============================================================

dist_classes = (
    df["Category_nome"]
    .value_counts()
    .rename_axis("Classe")
    .reset_index(name="Quantidade")
)

dist_classes["Percentual"] = 100 * dist_classes["Quantidade"] / len(df)

display(dist_classes)

plt.figure(figsize=(12, 5))
sns.countplot(
    data=df,
    y="Category_nome",
    order=df["Category_nome"].value_counts().index,
    palette="viridis"
)
plt.title("Distribuição das classes")
plt.xlabel("Quantidade de documentos")
plt.ylabel("Classe")
plt.show()

,Classe,Quantidade,Percentual
0,Recurso Extraordinário (RE),12876,56.772487
1,Agravo de Recurso Extraordinário (ARE),3529,15.559965
2,Sentença,2903,12.799824
3,Acórdão,665,2.932099
4,Despacho,439,1.935626


/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/1894718062.py:17: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/1894718062.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [438]:
# ============================================================
# 2.1 Verificação de desbalanceamento
# ============================================================

maior_classe = dist_classes["Quantidade"].max()
menor_classe = dist_classes["Quantidade"].min()
razao_desbalanceamento = maior_classe / menor_classe

print(f"Maior classe: {maior_classe} documentos")
print(f"Menor classe: {menor_classe} documentos")
print(f"Razão de desbalanceamento: {razao_desbalanceamento:.2f}")

if razao_desbalanceamento > 2:
    print("Há indícios de desbalanceamento relevante entre as classes.")
else:
    print("A distribuição das classes não parece muito desbalanceada.")

Maior classe: 12876 documentos
Menor classe: 439 documentos
Razão de desbalanceamento: 29.33
Há indícios de desbalanceamento relevante entre as classes.


In [439]:
# ============================================================
# 3. Análise textual: tamanho dos documentos
# ============================================================

def contar_palavras(texto):
    return len(str(texto).split())

def contar_caracteres(texto):
    return len(str(texto))

def contar_sentencas(texto):
    return len(re.findall(r"[.!?]+", str(texto)))

df["qtd_caracteres"] = df["Body"].apply(contar_caracteres)
df["qtd_palavras"] = df["Body"].apply(contar_palavras)
df["qtd_sentencas"] = df["Body"].apply(contar_sentencas)

display(df[["qtd_caracteres", "qtd_palavras", "qtd_sentencas"]].describe())

,qtd_caracteres,qtd_palavras,qtd_sentencas
count,22680.000000,22680.000000,22680.0
mean,1758.236772,216.426058,0.0
std,750.062347,93.088142,0.0
min,4.000000,1.000000,0.0
25%,1293.000000,159.000000,0.0
50%,1715.000000,209.000000,0.0
75%,2177.000000,268.000000,0.0
max,6867.000000,844.000000,0.0


In [440]:
# ============================================================
# 3.1 Distribuição do tamanho dos textos
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["qtd_palavras"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Distribuição da quantidade de palavras")
axes[0].set_xlabel("Quantidade de palavras")

sns.boxplot(data=df, x="qtd_palavras", ax=axes[1])
axes[1].set_title("Boxplot da quantidade de palavras")
axes[1].set_xlabel("Quantidade de palavras")

plt.tight_layout()
plt.show()

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/2961410962.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [441]:
# ============================================================
# 3.2 Tamanho dos textos por classe
# ============================================================

estatisticas_por_classe = (
    df.groupby("Category_nome")[["qtd_caracteres", "qtd_palavras", "qtd_sentencas"]]
    .agg(["mean", "median", "std", "min", "max"])
)

display(estatisticas_por_classe)

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,
    y="Category_nome",
    x="qtd_palavras",
    palette="viridis"
)
plt.title("Distribuição do tamanho dos textos por classe")
plt.xlabel("Quantidade de palavras")
plt.ylabel("Classe")
plt.show()

qtd_caracteres                      \
                                                 mean  median         std   
Category_nome                                                               
Acórdão                                   2011.908271  2119.0  907.395586   
Agravo de Recurso Extraordinário (ARE)    1578.096345  1456.0  735.338974   
Despacho                                  1195.735763  1154.0  852.372715   
Recurso Extraordinário (RE)               1746.595216  1712.0  704.201052   
Sentença                                  2046.951085  1931.0  771.020978   

                                                  qtd_palavras         \
                                        min   max         mean median   
Category_nome                                                           
Acórdão                                 155  4798   251.539850  262.0   
Agravo de Recurso Extraordinário (ARE)    4  5702   195.200057  181.0   
Despacho                                113  6867   149.036446  152.0   
Recurso Extraordinário (RE)              33  5923   213.882883  206.0   
Sentença                                 94  6448   254.391319  243.0   

                                                            qtd_sentencas  \
                                               std min  max          mean   
Category_nome                                                               
Acórdão                                 114.338256  19  576           0.0   
Agravo de Recurso Extraordinário (ARE)   90.091137   1  685           0.0   
Despacho                                102.660404  14  844           0.0   
Recurso Extraordinário (RE)              87.938303   4  781           0.0   
Sentença                                 94.121400  13  837           0.0   

                                                            
                                       median  std min max  
Category_nome                                               
Acórdão                                   0.0  0.0   0   0  
Agravo de Recurso Extraordinário (ARE)    0.0  0.0   0   0  
Despacho                                  0.0  0.0   0   0  
Recurso Extraordinário (RE)               0.0  0.0   0   0  
Sentença                                  0.0  0.0   0   0

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/390811310.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/390811310.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [442]:
# ============================================================
# 5. Palavras mais frequentes no corpus
# ============================================================

stopwords_pt = {
    "a", "o", "os", "as", "um", "uma", "uns", "umas",
    "de", "do", "da", "dos", "das", "em", "no", "na", "nos", "nas",
    "por", "para", "com", "sem", "sob", "sobre", "entre",
    "e", "ou", "mas", "que", "se", "ao", "aos", "à", "às",
    "é", "são", "foi", "ser", "sua", "seu", "suas", "seus",
    "não", "sim", "também", "como", "mais", "menos",
    "este", "esta", "estes", "estas", "esse", "essa", "isso",
    "aquele", "aquela", "aqueles", "aquelas",
    "lhe", "lhes", "me", "te", "nos", "vos"
}

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=list(stopwords_pt),
    min_df=5,
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{3,}\b"
)

X_counts = vectorizer.fit_transform(df["Body"].astype(str))

palavras = vectorizer.get_feature_names_out()
frequencias = np.asarray(X_counts.sum(axis=0)).ravel()

freq_df = pd.DataFrame({
    "palavra": palavras,
    "frequencia": frequencias
}).sort_values("frequencia", ascending=False)

display(freq_df.head(30))

plt.figure(figsize=(12, 6))
sns.barplot(
    data=freq_df.head(20),
    x="frequencia",
    y="palavra",
    palette="mako"
)
plt.title("20 palavras mais frequentes no corpus")
plt.xlabel("Frequência")
plt.ylabel("Palavra")
plt.show()

,palavra,frequencia
12484,rio,54583
5991,federal,33760
12461,ria,29127
11672,recurso,27826
8292,lei,25684
10160,pela,24809
6501,geral,23459
5970,fazenda,20255
1636,benefã,19814
3160,contribuiã,19636


/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/1095828495.py:37: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/1095828495.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [443]:
# ============================================================
# 6. Vocabulário da base
# ============================================================

vocabulario_total = len(palavras)

print(f"Tamanho do vocabulário considerando min_df=5: {vocabulario_total}")

print("\nPalavras mais frequentes:")
display(freq_df.head(20))

print("\nPalavras menos frequentes dentro do filtro min_df=5:")
display(freq_df.tail(20))

Tamanho do vocabulário considerando min_df=5: 14690

Palavras mais frequentes:


,palavra,frequencia
12484,rio,54583
5991,federal,33760
12461,ria,29127
11672,recurso,27826
8292,lei,25684
10160,pela,24809
6501,geral,23459
5970,fazenda,20255
1636,benefã,19814
3160,contribuiã,19636



Palavras menos frequentes dentro do filtro min_df=5:


,palavra,frequencia
13685,tir,5
1529,aymore,5
6738,hartmann,5
11701,redundantes,5
12620,safatle,5
1490,autotutela,5
6802,hominis,5
4448,disponibiliza,5
6797,homenageia,5
6793,hohmann,5


In [444]:
# ============================================================
# 7. Termos jurídicos frequentes
# ============================================================

termos_juridicos = [
    "recurso", "extraordinário", "agravo", "acórdão", "sentença",
    "despacho", "relator", "ministro", "tribunal", "supremo",
    "federal", "constitucional", "processo", "decisão", "julgamento",
    "jurisprudência", "embargos", "apelação", "recorrente", "recorrido",
    "provimento", "improvimento", "admissibilidade", "mérito"
]

textos_minusculos = df["Body"].astype(str).str.lower()

contagem_termos = []

for termo in termos_juridicos:
    padrao = r"\b" + re.escape(termo.lower()) + r"\b"
    freq = textos_minusculos.str.count(padrao).sum()
    contagem_termos.append((termo, freq))

termos_df = pd.DataFrame(contagem_termos, columns=["termo", "frequencia"])
termos_df = termos_df.sort_values("frequencia", ascending=False)

display(termos_df)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=termos_df,
    x="frequencia",
    y="termo",
    palette="crest"
)
plt.title("Frequência de termos jurídicos selecionados")
plt.xlabel("Frequência")
plt.ylabel("Termo jurídico")
plt.show()

,termo,frequencia
10,federal,33760
0,recurso,27826
11,constitucional,17501
8,tribunal,14778
12,processo,9187
2,agravo,7898
9,supremo,7192
14,julgamento,5858
6,relator,5303
18,recorrente,5287


/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/1734049659.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/1734049659.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [445]:
# ============================================================
# 8. Palavras mais comuns por classe
# ============================================================

def palavras_mais_frequentes_por_classe(df, classe, n=15):
    textos = df.loc[df["Category_nome"] == classe, "Body"].astype(str)

    vectorizer_classe = CountVectorizer(
        lowercase=True,
        stop_words=list(stopwords_pt),
        min_df=2,
        token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{3,}\b"
    )

    X = vectorizer_classe.fit_transform(textos)

    palavras_classe = vectorizer_classe.get_feature_names_out()
    frequencias_classe = np.asarray(X.sum(axis=0)).ravel()

    resultado = pd.DataFrame({
        "palavra": palavras_classe,
        "frequencia": frequencias_classe
    }).sort_values("frequencia", ascending=False)

    return resultado.head(n)


for classe in df["Category_nome"].dropna().unique():
    print(f"\nClasse: {classe}")
    display(palavras_mais_frequentes_por_classe(df, classe, n=15))


Classe: Recurso Extraordinário (RE)


,palavra,frequencia
14640,rio,31769
6951,federal,19590
7556,geral,16238
14612,ria,16213
6927,fazenda,16032
9753,lei,15848
12954,procuradoria,14611
10812,nacional,14147
13740,recurso,13734
11949,pela,13665



Classe: Agravo de Recurso Extraordinário (ARE)


,palavra,frequencia
10371,rio,8974
9717,recurso,7294
4999,federal,4951
4870,extraordinã,4304
2992,decisã,4060
10351,ria,3719
8456,pela,3446
8459,pelo,3049
5412,geral,2907
441,agravo,2834



Classe: Despacho


,palavra,frequencia
1887,recurso,1426
2013,rio,1331
81,agravo,917
924,extraordinã,886
564,decisã,778
946,federal,743
1886,recursal,677
2254,turma,654
2247,tribunal,572
2092,serã,525



Classe: Sentença


,palavra,frequencia
7344,rio,5786
1851,contribuiã,5067
7330,ria,4854
4833,lei,4640
966,benefã,4497
5930,pela,4185
3499,federal,4042
7547,servidores,3568
5856,parte,3559
1293,cios,3443



Classe: Acórdão


,palavra,frequencia
344,benefã,1294
2714,rio,1247
450,cios,1126
2707,ria,1073
1247,federal,1041
2552,recurso,1041
2178,parte,938
2779,sentenã,870
3037,turma,854
3067,valor,813


In [446]:
# ============================================================
# 8.1 Gráficos das palavras mais comuns por classe
# ============================================================

classes = df["Category_nome"].dropna().unique()

for classe in classes:
    top_palavras = palavras_mais_frequentes_por_classe(df, classe, n=15)

    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=top_palavras,
        x="frequencia",
        y="palavra",
        palette="viridis"
    )
    plt.title(f"Palavras mais frequentes - {classe}")
    plt.xlabel("Frequência")
    plt.ylabel("Palavra")
    plt.show()

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/3355619004.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/3355619004.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/3355619004.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/3355619004.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/3355619004.py:11: FutureWarning: 

In [447]:
# ============================================================
# 9. N-gramas mais frequentes
# ============================================================

def ngramas_mais_frequentes(textos, ngram_range=(2, 2), top_n=20):
    vectorizer_ngram = CountVectorizer(
        lowercase=True,
        stop_words=list(stopwords_pt),
        ngram_range=ngram_range,
        min_df=3,
        token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{3,}\b"
    )

    X = vectorizer_ngram.fit_transform(textos.astype(str))

    termos = vectorizer_ngram.get_feature_names_out()
    frequencias = np.asarray(X.sum(axis=0)).ravel()

    return (
        pd.DataFrame({"termo": termos, "frequencia": frequencias})
        .sort_values("frequencia", ascending=False)
        .head(top_n)
    )


bigramas = ngramas_mais_frequentes(df["Body"], ngram_range=(2, 2), top_n=20)
trigramas = ngramas_mais_frequentes(df["Body"], ngram_range=(3, 3), top_n=20)

print("Bigramas mais frequentes:")
display(bigramas)

print("Trigramas mais frequentes:")
display(trigramas)

Bigramas mais frequentes:


,termo,frequencia
49353,fazenda nacional,13705
48322,extraordinã rio,13332
102659,recurso extraordinã,12709
94785,princã pio,12042
13955,benefã cios,10298
24603,constituiã federal,9879
13954,benefã cio,9509
72610,matã ria,8103
95686,procuradoria geral,7712
94227,previdenciã ria,7664


Trigramas mais frequentes:


,termo,frequencia
126624,recurso extraordinã rio,12702
148824,supremo tribunal federal,6943
31074,contribuiã previdenciã ria,6008
66993,geral fazenda nacional,5736
118666,procuradoria geral fazenda,5731
59697,fazenda nacional procuradoria,5450
91170,ministã rio fazenda,5354
136518,rio fazenda procuradoria,5342
59744,fazenda procuradoria geral,5320
2846,advocacia geral uniã,4607


In [448]:
# ============================================================
# 9.1 Gráficos dos n-gramas mais frequentes
# ============================================================

plt.figure(figsize=(12, 6))
sns.barplot(
    data=bigramas,
    x="frequencia",
    y="termo",
    palette="rocket"
)
plt.title("Bigramas mais frequentes")
plt.xlabel("Frequência")
plt.ylabel("Bigrama")
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=trigramas,
    x="frequencia",
    y="termo",
    palette="flare"
)
plt.title("Trigramas mais frequentes")
plt.xlabel("Frequência")
plt.ylabel("Trigrama")
plt.show()

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/2766490457.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/2766490457.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/2766490457.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/2766490457.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [449]:
# ============================================================
# 10. Resumo final da análise exploratória
# ============================================================

print("Resumo da base:")
print(f"- Total de documentos: {len(df)}")
print(f"- Total de classes: {df['Category'].nunique()}")
print(f"- Vocabulário aproximado observado na EDA: {vocabulario_total} termos")
print(f"- Média de palavras por documento: {df['qtd_palavras'].mean():.2f}")
print(f"- Mediana de palavras por documento: {df['qtd_palavras'].median():.2f}")
print(f"- Maior documento: {df['qtd_palavras'].max()} palavras")
print(f"- Menor documento: {df['qtd_palavras'].min()} palavras")
print(f"- Razão de desbalanceamento: {razao_desbalanceamento:.2f}")

Resumo da base:
- Total de documentos: 22680
- Total de classes: 6
- Vocabulário aproximado observado na EDA: 14690 termos
- Média de palavras por documento: 216.43
- Mediana de palavras por documento: 209.00
- Maior documento: 844 palavras
- Menor documento: 1 palavras
- Razão de desbalanceamento: 29.33


EXEMPLO:
A análise exploratória foi realizada sobre a base VICTOR, composta por documentos jurídicos do STF classificados em cinco categorias: Acórdão, Agravo de Recurso Extraordinário, Despacho, Recurso Extraordinário e Sentença. Foram analisados o formato da base, valores ausentes, duplicatas, distribuição das classes, tamanho dos textos, vocabulário mais frequente, termos jurídicos recorrentes e diferenças textuais entre as categorias. Essa etapa é importante para compreender o corpus antes da aplicação de técnicas de pré-processamento, vetorização e modelagem preditiva.

---
### Pré-processamento

Nesta seção, as funções da etapa de pré-processamento dos dados devem ser implementadas e aplicadas (se necessário).

In [450]:
# ============================================================
# 1. Mapeamento das classes
# ============================================================

mapa_classes = {
    0: "Acórdão",
    1: "Agravo de Recurso Extraordinário (ARE)",
    2: "Despacho",
    3: "Recurso Extraordinário (RE)",
    4: "Sentença"
}

df["Category_nome"] = df["Category"].map(mapa_classes)

df[["Category", "Category_nome"]].head()

,Category,Category_nome
0,3,Recurso Extraordinário (RE)
1,3,Recurso Extraordinário (RE)
2,1,Agravo de Recurso Extraordinário (ARE)
3,3,Recurso Extraordinário (RE)
4,3,Recurso Extraordinário (RE)


In [451]:
# ============================================================
# 2. Tratamento de valores ausentes
# ============================================================

print("Valores nulos antes do tratamento:")
print(df.isnull().sum())

# Remove linhas sem texto ou sem classe
df = df.dropna(subset=["Body", "Category"]).copy()

# Garante que o texto está no formato string
df["Body"] = df["Body"].astype(str)

print("\nValores nulos depois do tratamento:")
print(df.isnull().sum())

Valores nulos antes do tratamento:
Id                   0
Body                 0
Category             0
Category_nome     2268
qtd_caracteres       0
qtd_palavras         0
qtd_sentencas        0
dtype: int64

Valores nulos depois do tratamento:
Id                   0
Body                 0
Category             0
Category_nome     2268
qtd_caracteres       0
qtd_palavras         0
qtd_sentencas        0
dtype: int64


In [452]:
# ============================================================
# 3. Remoção de duplicatas
# ============================================================

print("Tamanho antes da remoção de duplicatas:", df.shape)

# Remove documentos repetidos pelo identificador, se houver
df = df.drop_duplicates(subset=["Id"]).copy()

# Remove textos exatamente iguais, se houver
df = df.drop_duplicates(subset=["Body"]).copy()

print("Tamanho depois da remoção de duplicatas:", df.shape)

Tamanho antes da remoção de duplicatas: (22680, 7)
Tamanho depois da remoção de duplicatas: (11497, 7)


In [453]:
# ============================================================
# 4. Lista de stopwords em português
# ============================================================

stopwords_pt = {
    "a", "o", "os", "as", "um", "uma", "uns", "umas",
    "de", "do", "da", "dos", "das", "em", "no", "na", "nos", "nas",
    "por", "para", "com", "sem", "sob", "sobre", "entre",
    "e", "ou", "mas", "que", "se", "ao", "aos", "à", "às",
    "é", "são", "foi", "foram", "era", "eram", "ser", "estar",
    "sua", "seu", "suas", "seus", "meu", "minha", "meus", "minhas",
    "não", "sim", "também", "como", "mais", "menos", "muito", "muita",
    "este", "esta", "estes", "estas", "esse", "essa", "esses", "essas",
    "isso", "isto", "aquele", "aquela", "aqueles", "aquelas",
    "lhe", "lhes", "me", "te", "nos", "vos",
    "já", "ainda", "então", "após", "antes", "quando", "onde",
    "qual", "quais", "quem", "cujo", "cuja", "cujos", "cujas"
}

In [454]:
# ============================================================
# 5. Funções de pré-processamento textual
# ============================================================

def remover_acentos(texto):
    """
    Remove acentos e sinais diacríticos.
    Exemplo: 'acórdão' -> 'acordao'
    """
    texto = unicodedata.normalize("NFKD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    return texto


def normalizar_texto(texto, remover_acentuacao=False):
    """
    Aplica normalização básica:
    - transforma em minúsculas
    - remove quebras de linha excessivas
    - remove espaços duplicados
    - opcionalmente remove acentos
    """
    texto = str(texto).lower()
    texto = re.sub(r"\n+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    if remover_acentuacao:
        texto = remover_acentos(texto)

    return texto


def remover_ruidos_ocr(texto):
    """
    Remove ruídos comuns de textos extraídos por OCR,
    como caracteres soltos, símbolos repetidos e espaços excessivos.
    """
    texto = re.sub(r"[_\-]{2,}", " ", texto)
    texto = re.sub(r"[^\w\sáàâãéèêíïóôõöúçñ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def remover_numeros(texto):
    """
    Remove números isolados ou sequências numéricas.
    Em textos jurídicos, isso pode remover datas, artigos e números processuais.
    Use com cuidado, dependendo do experimento.
    """
    texto = re.sub(r"\d+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def tokenizar(texto):
    """
    Separa o texto em tokens.
    """
    return texto.split()


def remover_stopwords(tokens):
    """
    Remove palavras muito frequentes que normalmente carregam pouca informação semântica.
    """
    return [token for token in tokens if token not in stopwords_pt]


def remover_tokens_curtos(tokens, tamanho_minimo=3):
    """
    Remove tokens muito curtos, como letras isoladas.
    """
    return [token for token in tokens if len(token) >= tamanho_minimo]


def reconstruir_texto(tokens):
    """
    Reconstrói o texto a partir dos tokens processados.
    """
    return " ".join(tokens)

In [455]:
# ============================================================
# 6. Pipeline de pré-processamento
# ============================================================

def preprocessar_texto(
    texto,
    remover_acentuacao=False,
    remover_nums=True,
    remover_stops=True,
    tamanho_minimo_token=3
):
    """
    Pipeline completo de pré-processamento textual.
    """
    texto = normalizar_texto(texto, remover_acentuacao=remover_acentuacao)
    texto = remover_ruidos_ocr(texto)

    if remover_nums:
        texto = remover_numeros(texto)

    tokens = tokenizar(texto)

    if remover_stops:
        tokens = remover_stopwords(tokens)

    tokens = remover_tokens_curtos(tokens, tamanho_minimo=tamanho_minimo_token)

    texto_processado = reconstruir_texto(tokens)

    return texto_processado

In [456]:
# ============================================================
# 7. Aplicação do pré-processamento
# ============================================================

df["Body_preprocessado"] = df["Body"].apply(
    lambda texto: preprocessar_texto(
        texto,
        remover_acentuacao=False,
        remover_nums=True,
        remover_stops=True,
        tamanho_minimo_token=3
    )
)

df[["Body", "Body_preprocessado"]].head()

,Body,Body_preprocessado
0,"{""conclusÃ£o diante exposto recorrente espera ...",conclusã diante exposto recorrente espera conf...
1,"{""ARTIGO_55 gecen a gacen serÃ£o devidas aos t...",artigo_ gecen gacen serã devidas titulares emp...
2,"{""seguir desacerto decisÃ£o que negou seguimen...",seguir desacerto decisã negou seguimento recur...
3,"{""advocacia geral uniÃ£o procuradoria uniÃ£o p...",advocacia geral uniã procuradoria uniã paraã g...
4,"{""ministÃ©rio fazenda procuradoria geral fazen...",ministã rio fazenda procuradoria geral fazenda...


In [457]:
# ============================================================
# 8. Comparação antes e depois do pré-processamento
# ============================================================

df["qtd_palavras_original"] = df["Body"].apply(lambda x: len(str(x).split()))
df["qtd_palavras_preprocessado"] = df["Body_preprocessado"].apply(lambda x: len(str(x).split()))

comparacao = df[[
    "qtd_palavras_original",
    "qtd_palavras_preprocessado"
]].describe()

display(comparacao)

,qtd_palavras_original,qtd_palavras_preprocessado
count,11497.000000,11497.000000
mean,217.138645,192.090719
std,94.175336,82.216726
min,1.000000,0.000000
25%,155.000000,138.000000
50%,214.000000,190.000000
75%,274.000000,242.000000
max,844.000000,768.000000


In [458]:
# ============================================================
# 9. Visualização da redução dos textos
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(df["qtd_palavras_original"], bins=50, label="Original", color="steelblue", alpha=0.5)
sns.histplot(df["qtd_palavras_preprocessado"], bins=50, label="Pré-processado", color="orange", alpha=0.5)

plt.title("Comparação do tamanho dos textos antes e depois do pré-processamento")
plt.xlabel("Quantidade de palavras")
plt.ylabel("Frequência")
plt.legend()
plt.show()

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/826137416.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [459]:
# ============================================================
# 10. Exemplos de textos antes e depois
# ============================================================

for i in range(3):
    print("=" * 100)
    print("TEXTO ORIGINAL:")
    print(df["Body"].iloc[i][:1000])

    print("\nTEXTO PRÉ-PROCESSADO:")
    print(df["Body_preprocessado"].iloc[i][:1000])

TEXTO ORIGINAL:
{"conclusÃ£o diante exposto recorrente espera confia que recurso ora arrazoado interposto com fulcro ARTIGO_102 iii constituiÃ§Ã£o federal serÃ¡ admitido conhecido provido fim que seja anulado v acÃ³rdÃ£o impugnado assim nÃ£o entender que seja inteiramente reformado por violaÃ§Ã£o flagrante aos dispositivos constitucionais suscitados nas razÃµes recursais espera deferimento rio janeiro de outubro alexandre simÃµes camara silva procurador estado"}

TEXTO PRÉ-PROCESSADO:
conclusã diante exposto recorrente espera confia recurso ora arrazoado interposto fulcro artigo_ iii constituiã federal serã admitido conhecido provido fim seja anulado acã³rdã impugnado assim entender seja inteiramente reformado violaã flagrante dispositivos constitucionais suscitados razãµes recursais espera deferimento rio janeiro outubro alexandre simãµes camara silva procurador estado
TEXTO ORIGINAL:
{"ARTIGO_55 gecen a gacen serÃ£o devidas aos titulares dos empregos cargos pÃºblicos que tratam ARTIG

In [460]:
# ============================================================
# 11. Salvamento da base pré-processada
# ============================================================

df.to_csv("train_preprocessado.csv", index=False)

print("Arquivo salvo como train_preprocessado.csv")

Arquivo salvo como train_preprocessado.csv


EXEMPLO:
Nesta etapa, foram implementadas funções de pré-processamento textual com o objetivo de padronizar os documentos antes da representação computacional e do treinamento dos modelos. O processo incluiu tratamento de valores ausentes, remoção de duplicatas, normalização de caixa, remoção de ruídos de OCR, remoção de números, tokenização, remoção de stopwords e remoção de tokens muito curtos. A base resultante contém uma nova coluna, Body_preprocessado, que será utilizada nas etapas seguintes de vetorização e modelagem.

In [461]:
# ============================================================
# Pseudo-rotulagem - separação entre rotulados e não rotulados
# ============================================================

classes_validas = [0, 1, 2, 3, 4]

df_rotulado = df[df["Category"].isin(classes_validas)].copy()
df_nao_rotulado = df[df["Category"] == -1].copy()

print("Amostras rotuladas:", df_rotulado.shape)
print("Amostras não rotuladas:", df_nao_rotulado.shape)

TEXT_COL = "Body_preprocessado" if "Body_preprocessado" in df.columns else "Body"

X_rotulado = df_rotulado[TEXT_COL].astype(str)
y_rotulado = df_rotulado["Category"]

X_nao_rotulado = df_nao_rotulado[TEXT_COL].astype(str)

Amostras rotuladas: (10316, 10)
Amostras não rotuladas: (1181, 10)


In [462]:
# ============================================================
# Pseudo-rotulagem - treinamento do modelo auxiliar
# ============================================================

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_pseudo = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_rotulado_tfidf = tfidf_pseudo.fit_transform(X_rotulado)

modelo_pseudo = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

modelo_pseudo.fit(X_rotulado_tfidf, y_rotulado)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [463]:
# ============================================================
# Pseudo-rotulagem - inferência das classes desconhecidas
# ============================================================

if len(df_nao_rotulado) > 0:
    X_nao_rotulado_tfidf = tfidf_pseudo.transform(X_nao_rotulado)

    probabilidades = modelo_pseudo.predict_proba(X_nao_rotulado_tfidf)
    pseudo_labels = modelo_pseudo.classes_[np.argmax(probabilidades, axis=1)]
    confiancas = np.max(probabilidades, axis=1)

    df_nao_rotulado["pseudo_Category"] = pseudo_labels
    df_nao_rotulado["pseudo_confianca"] = confiancas

    display(df_nao_rotulado[["Id", "pseudo_Category", "pseudo_confianca"]].head())

    print("Distribuição das pseudo-classes:")
    print(df_nao_rotulado["pseudo_Category"].value_counts().sort_index())
else:
    print("Não há amostras com Category = -1.")

,Id,pseudo_Category,pseudo_confianca
34,9716,4,0.448081
35,25112,3,0.763383
44,11274,2,0.647456
46,20985,4,0.717429
119,23109,3,0.879257


Distribuição das pseudo-classes:
pseudo_Category
0     40
1    289
2     26
3    602
4    224
Name: count, dtype: int64


In [464]:
# ============================================================
# Pseudo-rotulagem - seleção das previsões confiáveis
# ============================================================

LIMIAR_CONFIANCA = 0.80

if len(df_nao_rotulado) > 0:
    df_pseudo_confiavel = df_nao_rotulado[
        df_nao_rotulado["pseudo_confianca"] >= LIMIAR_CONFIANCA
    ].copy()

    df_pseudo_confiavel["Category"] = df_pseudo_confiavel["pseudo_Category"]

else:
    df_pseudo_confiavel = pd.DataFrame(columns=df.columns)

print("Pseudo-rótulos aceitos:", df_pseudo_confiavel.shape[0])
print("Pseudo-rótulos rejeitados:", df_nao_rotulado.shape[0] - df_pseudo_confiavel.shape[0])

if df_pseudo_confiavel.shape[0] > 0:
    print("\nDistribuição dos pseudo-rótulos aceitos:")
    print(df_pseudo_confiavel["Category"].value_counts().sort_index())

Pseudo-rótulos aceitos: 416
Pseudo-rótulos rejeitados: 765

Distribuição dos pseudo-rótulos aceitos:
Category
0     29
1    132
2     13
3    128
4    114
Name: count, dtype: int64


In [465]:
# ============================================================
# Pseudo-rotulagem - criação da base final de modelagem
# ============================================================

colunas_modelagem = df_rotulado.columns.tolist()

df_modelagem = pd.concat(
    [
        df_rotulado[colunas_modelagem],
        df_pseudo_confiavel[colunas_modelagem]
    ],
    ignore_index=True
)

df_modelagem["Category"] = df_modelagem["Category"].astype(int)

print("Base original:", df.shape)
print("Base rotulada:", df_rotulado.shape)
print("Base expandida para modelagem:", df_modelagem.shape)

print("\nDistribuição final das classes:")
print(df_modelagem["Category"].value_counts().sort_index())

Base original: (11497, 10)
Base rotulada: (10316, 10)
Base expandida para modelagem: (10732, 10)

Distribuição final das classes:
Category
0     343
1    2646
2     251
3    5733
4    1759
Name: count, dtype: int64


---
### Experimento

Nesta seção, o experimento deve ser conduzido, utilizando os protocolos experimentais padrões e testando diferentes modelos.

A avaliação dos modelos foi realizada por meio de métricas amplamente utilizadas em tarefas de classificação: accuracy, precision, recall e F1-score. Como a base pode apresentar desbalanceamento entre as classes, a métrica principal adotada foi o Macro F1, pois ela calcula a média do desempenho entre as classes sem ponderar pela quantidade de exemplos de cada uma. Dessa forma, modelos que apresentam bom desempenho apenas nas classes majoritárias são penalizados. Além disso, foram analisadas matrizes de confusão e métricas por classe, permitindo observar quais categorias jurídicas foram mais confundidas pelos classificadores.




In [466]:
# ============================================================
# 1. Configuração inicial do experimento
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

RANDOM_STATE = 42

TEXT_COL = "Body_preprocessado" if "Body_preprocessado" in df.columns else "Body"
TARGET_COL = "Category"

X = df_modelagem[TEXT_COL].astype(str)
y = df_modelagem[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("Coluna de texto usada:", TEXT_COL)

Treino: (8585,)
Teste: (2147,)
Coluna de texto usada: Body_preprocessado


In [467]:
# ============================================================
# 1. Função correta para registrar resultados
# ============================================================

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

resultados = []
predicoes_modelos = {}

def registrar_resultado(nome_modelo, representacao, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    resultados.append({
        "modelo": nome_modelo,
        "representacao": representacao,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    })

    nome_experimento = f"{nome_modelo} - {representacao}"
    predicoes_modelos[nome_experimento] = y_pred

    print(f"Resultado registrado: {nome_experimento}")

In [468]:
# ============================================================
# 3. Representação textual: Bag of Words
# ============================================================

from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print("Bag of Words treino:", X_train_bow.shape)
print("Bag of Words teste:", X_test_bow.shape)

Bag of Words treino: (8585, 30000)
Bag of Words teste: (2147, 30000)


In [469]:
# ============================================================
# 4. Representação textual: TF-IDF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF treino:", X_train_tfidf.shape)
print("TF-IDF teste:", X_test_tfidf.shape)

TF-IDF treino: (8585, 50000)
TF-IDF teste: (2147, 50000)


In [470]:
# ============================================================
# 5. Representação textual densa: TF-IDF + SVD
# ============================================================

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

svd = TruncatedSVD(
    n_components=300,
    random_state=RANDOM_STATE
)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

scaler = StandardScaler()
X_train_svd = scaler.fit_transform(X_train_svd)
X_test_svd = scaler.transform(X_test_svd)

print("SVD treino:", X_train_svd.shape)
print("SVD teste:", X_test_svd.shape)
print("Variância explicada:", svd.explained_variance_ratio_.sum())

SVD treino: (8585, 300)
SVD teste: (2147, 300)
Variância explicada: 0.4009235201920316


In [471]:
# ============================================================
# 6. Modelos clássicos com Bag of Words
# ============================================================

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

modelos_bow = {
    "Naive Bayes": MultinomialNB(),
    "Regressão Logística": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "SVM Linear": LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

for nome, modelo in modelos_bow.items():
    modelo.fit(X_train_bow, y_train)
    y_pred = modelo.predict(X_test_bow)
    registrar_resultado(nome, "Bag of Words", y_test, y_pred)

Resultado registrado: Naive Bayes - Bag of Words
Resultado registrado: Regressão Logística - Bag of Words
Resultado registrado: SVM Linear - Bag of Words


/Users/annesakai/Downloads/PLN/template-implementacao/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [472]:
# ============================================================
# 7. Modelos clássicos com TF-IDF
# ============================================================

modelos_tfidf = {
    "Naive Bayes": MultinomialNB(),
    "Regressão Logística": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "SVM Linear": LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

for nome, modelo in modelos_tfidf.items():
    modelo.fit(X_train_tfidf, y_train)
    y_pred = modelo.predict(X_test_tfidf)
    registrar_resultado(nome, "TF-IDF", y_test, y_pred)

Resultado registrado: Naive Bayes - TF-IDF
Resultado registrado: Regressão Logística - TF-IDF
Resultado registrado: SVM Linear - TF-IDF


In [473]:
# ============================================================
# 8. Modelo neural simples com representação densa
# ============================================================

from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=20,
    random_state=RANDOM_STATE,
    early_stopping=True
)

mlp.fit(X_train_svd, y_train)

y_pred_mlp = mlp.predict(X_test_svd)

registrar_resultado(
    "MLP",
    "TF-IDF + SVD",
    y_test,
    y_pred_mlp
)

Resultado registrado: MLP - TF-IDF + SVD


/Users/annesakai/Downloads/PLN/template-implementacao/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


In [474]:
# ============================================================
# 12. Representação densa: TF-IDF + SVD
# ============================================================

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

svd = TruncatedSVD(
    n_components=300,
    random_state=RANDOM_STATE
)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

scaler = StandardScaler()

X_train_svd = scaler.fit_transform(X_train_svd)
X_test_svd = scaler.transform(X_test_svd)

print("TF-IDF + SVD treino:", X_train_svd.shape)
print("TF-IDF + SVD teste:", X_test_svd.shape)
print(f"Variância explicada pelo SVD: {svd.explained_variance_ratio_.sum():.4f}")

TF-IDF + SVD treino: (8585, 300)
TF-IDF + SVD teste: (2147, 300)
Variância explicada pelo SVD: 0.4009


In [475]:
# ============================================================
# 13. Modelo neural simples: MLP com TF-IDF + SVD
# ============================================================

from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=20,
    random_state=RANDOM_STATE,
    early_stopping=True
)

mlp.fit(X_train_svd, y_train)

y_pred_mlp = mlp.predict(X_test_svd)

registrar_resultado(
    nome_modelo="MLP",
    representacao="TF-IDF + SVD",
    y_true=y_test,
    y_pred=y_pred_mlp
)

Resultado registrado: MLP - TF-IDF + SVD


/Users/annesakai/Downloads/PLN/template-implementacao/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


---
### Análise dos Resultados

Nesta seção, os resultados devem ser exibidos através de tabelas e gráficos, comparados e profundamente analisados.

In [476]:
# ============================================================
# Reconstrução completa dos resultados com Macro F1
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Modelos salvos em predicoes_modelos:")
print(list(predicoes_modelos.keys()))

resultados = []

for nome_experimento, y_pred in predicoes_modelos.items():

    nome_modelo, representacao = nome_experimento.split(" - ", 1)

    accuracy = accuracy_score(y_test, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    resultados.append({
        "modelo": nome_modelo,
        "representacao": representacao,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    })

resultados_df = pd.DataFrame(resultados)

resultados_df = resultados_df.sort_values(
    by="f1_macro",
    ascending=False
).reset_index(drop=True)

display(resultados_df)

Modelos salvos em predicoes_modelos:
['Naive Bayes - Bag of Words', 'Regressão Logística - Bag of Words', 'SVM Linear - Bag of Words', 'Naive Bayes - TF-IDF', 'Regressão Logística - TF-IDF', 'SVM Linear - TF-IDF', 'MLP - TF-IDF + SVD']


,modelo,representacao,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,SVM Linear,TF-IDF,0.872846,0.872972,0.851289,0.860276,0.871413,0.872846,0.871342
1,MLP,TF-IDF + SVD,0.866791,0.876187,0.822448,0.846713,0.866201,0.866791,0.865687
2,Regressão Logística,Bag of Words,0.850489,0.848202,0.838153,0.842661,0.850632,0.850489,0.850485
3,Regressão Logística,TF-IDF,0.847694,0.834816,0.849589,0.839897,0.847868,0.847694,0.846824
4,SVM Linear,Bag of Words,0.824872,0.839394,0.805903,0.821306,0.826178,0.824872,0.825322
5,Naive Bayes,Bag of Words,0.762459,0.727206,0.791418,0.754944,0.774502,0.762459,0.764031
6,Naive Bayes,TF-IDF,0.770377,0.877472,0.510737,0.575470,0.807531,0.770377,0.749709


In [477]:
# ============================================================
# Organização dos resultados
# ============================================================

resultados_df = pd.DataFrame(resultados)

if "f1_macro" not in resultados_df.columns:
    raise ValueError(
        "A coluna f1_macro ainda não existe. "
        "Rode a célula de correção dos resultados ou rerode os experimentos "
        "com a função registrar_resultado atualizada."
    )

resultados_df = resultados_df.sort_values(
    by="f1_macro",
    ascending=False
).reset_index(drop=True)

display(resultados_df)

,modelo,representacao,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,SVM Linear,TF-IDF,0.872846,0.872972,0.851289,0.860276,0.871413,0.872846,0.871342
1,MLP,TF-IDF + SVD,0.866791,0.876187,0.822448,0.846713,0.866201,0.866791,0.865687
2,Regressão Logística,Bag of Words,0.850489,0.848202,0.838153,0.842661,0.850632,0.850489,0.850485
3,Regressão Logística,TF-IDF,0.847694,0.834816,0.849589,0.839897,0.847868,0.847694,0.846824
4,SVM Linear,Bag of Words,0.824872,0.839394,0.805903,0.821306,0.826178,0.824872,0.825322
5,Naive Bayes,Bag of Words,0.762459,0.727206,0.791418,0.754944,0.774502,0.762459,0.764031
6,Naive Bayes,TF-IDF,0.770377,0.877472,0.510737,0.575470,0.807531,0.770377,0.749709


In [478]:
# ============================================================
# 2. Ranking dos modelos por Macro F1
# ============================================================

ranking = resultados_df[[
    "modelo",
    "representacao",
    "accuracy",
    "f1_macro",
    "f1_weighted"
]].copy()

ranking["experimento"] = ranking["modelo"] + " - " + ranking["representacao"]

display(ranking)

,modelo,representacao,accuracy,f1_macro,f1_weighted,experimento
0,SVM Linear,TF-IDF,0.872846,0.860276,0.871342,SVM Linear - TF-IDF
1,MLP,TF-IDF + SVD,0.866791,0.846713,0.865687,MLP - TF-IDF + SVD
2,Regressão Logística,Bag of Words,0.850489,0.842661,0.850485,Regressão Logística - Bag of Words
3,Regressão Logística,TF-IDF,0.847694,0.839897,0.846824,Regressão Logística - TF-IDF
4,SVM Linear,Bag of Words,0.824872,0.821306,0.825322,SVM Linear - Bag of Words
5,Naive Bayes,Bag of Words,0.762459,0.754944,0.764031,Naive Bayes - Bag of Words
6,Naive Bayes,TF-IDF,0.770377,0.575470,0.749709,Naive Bayes - TF-IDF


In [479]:
# ============================================================
# 3. Gráfico comparativo: Accuracy, Macro F1 e Weighted F1
# ============================================================

resultados_long = resultados_df.melt(
    id_vars=["modelo", "representacao"],
    value_vars=["accuracy", "f1_macro", "f1_weighted"],
    var_name="metrica",
    value_name="valor"
)

resultados_long["experimento"] = (
    resultados_long["modelo"] + " - " + resultados_long["representacao"]
)

plt.figure(figsize=(14, 7))

sns.barplot(
    data=resultados_long,
    x="valor",
    y="experimento",
    hue="metrica"
)

plt.title("Comparação dos modelos por Accuracy, Macro F1 e Weighted F1")
plt.xlabel("Valor da métrica")
plt.ylabel("Experimento")
plt.xlim(0, 1)
plt.legend(title="Métrica")
plt.show()

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/2111954641.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [480]:
# ============================================================
# 4. Melhor modelo segundo Macro F1
# ============================================================

melhor_resultado = resultados_df.iloc[0]

melhor_experimento = (
    melhor_resultado["modelo"] + " - " + melhor_resultado["representacao"]
)

y_pred_melhor = predicoes_modelos[melhor_experimento]

print("Melhor experimento segundo Macro F1:")
display(melhor_resultado)

print("Nome do experimento:", melhor_experimento)

Melhor experimento segundo Macro F1:


modelo                SVM Linear
representacao             TF-IDF
accuracy                0.872846
precision_macro         0.872972
recall_macro            0.851289
f1_macro                0.860276
precision_weighted      0.871413
recall_weighted         0.872846
f1_weighted             0.871342
Name: 0, dtype: object

Nome do experimento: SVM Linear - TF-IDF


In [481]:
# ============================================================
# Relatório detalhado do melhor modelo
# ============================================================

from sklearn.metrics import classification_report

labels_classes = sorted(mapa_classes.keys())
nomes_classes = [mapa_classes[i] for i in labels_classes]

print(classification_report(
    y_test,
    y_pred_melhor,
    labels=labels_classes,
    target_names=nomes_classes,
    zero_division=0
))

                                        precision    recall  f1-score   support

                               Acórdão       0.84      0.88      0.86        69
Agravo de Recurso Extraordinário (ARE)       0.82      0.74      0.78       529
                              Despacho       0.90      0.76      0.83        50
           Recurso Extraordinário (RE)       0.88      0.91      0.89      1147
                              Sentença       0.93      0.96      0.94       352

                              accuracy                           0.87      2147
                             macro avg       0.87      0.85      0.86      2147
                          weighted avg       0.87      0.87      0.87      2147



In [482]:
# ============================================================
# 6. Matriz de confusão do melhor modelo
# ============================================================

matriz = confusion_matrix(y_test, y_pred_melhor)

labels_classes = [mapa_classes[i] for i in sorted(mapa_classes.keys())]

plt.figure(figsize=(10, 8))

sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels_classes,
    yticklabels=labels_classes
)

plt.title(f"Matriz de confusão - {melhor_experimento}")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.show()

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/2005272515.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [483]:
# ============================================================
# 7. Matriz de confusão normalizada
# ============================================================

matriz_normalizada = confusion_matrix(
    y_test,
    y_pred_melhor,
    normalize="true"
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    matriz_normalizada,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=labels_classes,
    yticklabels=labels_classes
)

plt.title(f"Matriz de confusão normalizada - {melhor_experimento}")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.show()

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/4143644541.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [484]:
# ============================================================
# 8. Métricas por classe do melhor modelo
# ============================================================

from sklearn.metrics import classification_report

relatorio_dict = classification_report(
    y_test,
    y_pred_melhor,
    labels=labels_classes,
    target_names=nomes_classes,
    output_dict=True,
    zero_division=0
)

metricas_classes = pd.DataFrame(relatorio_dict).transpose()

metricas_classes = metricas_classes.loc[nomes_classes, [
    "precision",
    "recall",
    "f1-score",
    "support"
]]

display(metricas_classes)

,precision,recall,f1-score,support
Acórdão,0.0,0.0,0.0,0.0
Agravo de Recurso Extraordinário (ARE),0.0,0.0,0.0,0.0
Despacho,0.0,0.0,0.0,0.0
Recurso Extraordinário (RE),0.0,0.0,0.0,0.0
Sentença,0.0,0.0,0.0,0.0


In [485]:
# ============================================================
# 9. Gráfico de F1-score por classe
# ============================================================

metricas_classes_plot = metricas_classes.reset_index().rename(
    columns={"index": "classe", "f1-score": "f1_score"}
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=metricas_classes_plot,
    x="f1_score",
    y="classe",
    palette="viridis"
)

plt.title(f"F1-score por classe - {melhor_experimento}")
plt.xlabel("F1-score")
plt.ylabel("Classe")
plt.xlim(0, 1)
plt.show()

/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/1675853211.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/var/folders/_9/r6bn2mcs5ssb600sxg44xdv00000gn/T/ipykernel_10460/1675853211.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [486]:
# ============================================================
# 10. Comparação entre Macro F1 e Accuracy
# ============================================================

comparacao_metricas = resultados_df[[
    "modelo",
    "representacao",
    "accuracy",
    "f1_macro",
    "f1_weighted"
]].copy()

comparacao_metricas["diferenca_accuracy_macro_f1"] = (
    comparacao_metricas["accuracy"] - comparacao_metricas["f1_macro"]
)

display(comparacao_metricas.sort_values(
    "diferenca_accuracy_macro_f1",
    ascending=False
))

,modelo,representacao,accuracy,f1_macro,f1_weighted,diferenca_accuracy_macro_f1
6,Naive Bayes,TF-IDF,0.770377,0.575470,0.749709,0.194908
1,MLP,TF-IDF + SVD,0.866791,0.846713,0.865687,0.020078
0,SVM Linear,TF-IDF,0.872846,0.860276,0.871342,0.012569
2,Regressão Logística,Bag of Words,0.850489,0.842661,0.850485,0.007828
3,Regressão Logística,TF-IDF,0.847694,0.839897,0.846824,0.007798
5,Naive Bayes,Bag of Words,0.762459,0.754944,0.764031,0.007515
4,SVM Linear,Bag of Words,0.824872,0.821306,0.825322,0.003566


In [487]:
# ============================================================
# 11. Interpretação automática inicial dos resultados
# ============================================================

melhor_accuracy = resultados_df.sort_values(
    "accuracy",
    ascending=False
).iloc[0]

melhor_macro_f1 = resultados_df.sort_values(
    "f1_macro",
    ascending=False
).iloc[0]

print("Melhor modelo por Accuracy:")
print(f"{melhor_accuracy['modelo']} - {melhor_accuracy['representacao']}")
print(f"Accuracy: {melhor_accuracy['accuracy']:.4f}")
print(f"Macro F1: {melhor_accuracy['f1_macro']:.4f}")

print("\nMelhor modelo por Macro F1:")
print(f"{melhor_macro_f1['modelo']} - {melhor_macro_f1['representacao']}")
print(f"Accuracy: {melhor_macro_f1['accuracy']:.4f}")
print(f"Macro F1: {melhor_macro_f1['f1_macro']:.4f}")

if (
    melhor_accuracy["modelo"] != melhor_macro_f1["modelo"]
    or melhor_accuracy["representacao"] != melhor_macro_f1["representacao"]
):
    print("\nObservação:")
    print(
        "O melhor modelo por accuracy não é o mesmo que o melhor modelo por Macro F1. "
        "Isso indica que a acurácia pode estar favorecendo classes majoritárias, "
        "enquanto o Macro F1 avalia melhor o desempenho médio entre todas as classes."
    )
else:
    print("\nObservação:")
    print(
        "O mesmo modelo obteve o melhor desempenho tanto em accuracy quanto em Macro F1. "
        "Isso sugere um desempenho mais consistente entre as classes."
    )

Melhor modelo por Accuracy:
SVM Linear - TF-IDF
Accuracy: 0.8728
Macro F1: 0.8603

Melhor modelo por Macro F1:
SVM Linear - TF-IDF
Accuracy: 0.8728
Macro F1: 0.8603

Observação:
O mesmo modelo obteve o melhor desempenho tanto em accuracy quanto em Macro F1. Isso sugere um desempenho mais consistente entre as classes.


In [488]:
# ============================================================
# 12. Comparação estatística simples entre os dois melhores modelos
# ============================================================

from scipy.stats import chi2

if len(resultados_df) >= 2:
    experimento_1 = resultados_df.iloc[0]["modelo"] + " - " + resultados_df.iloc[0]["representacao"]
    experimento_2 = resultados_df.iloc[1]["modelo"] + " - " + resultados_df.iloc[1]["representacao"]

    y_pred_1 = predicoes_modelos[experimento_1]
    y_pred_2 = predicoes_modelos[experimento_2]

    correto_1 = y_pred_1 == y_test
    correto_2 = y_pred_2 == y_test

    b = np.sum(correto_1 & ~correto_2)
    c = np.sum(~correto_1 & correto_2)

    if b + c == 0:
        print("Os dois modelos cometeram erros exatamente nos mesmos casos.")
    else:
        estatistica = (abs(b - c) - 1) ** 2 / (b + c)
        p_valor = 1 - chi2.cdf(estatistica, df=1)

        print("Comparação estatística entre os dois melhores modelos por Macro F1")
        print("Modelo 1:", experimento_1)
        print("Modelo 2:", experimento_2)

        print(f"\nb: casos em que o modelo 1 acertou e o modelo 2 errou = {b}")
        print(f"c: casos em que o modelo 1 errou e o modelo 2 acertou = {c}")

        print(f"\nEstatística de McNemar aproximada: {estatistica:.4f}")
        print(f"p-valor: {p_valor:.4f}")

        if p_valor < 0.05:
            print("\nConclusão: há diferença estatisticamente significativa entre os modelos.")
        else:
            print("\nConclusão: não há evidência estatística suficiente de diferença significativa entre os modelos.")
else:
    print("É necessário ter pelo menos dois modelos avaliados para aplicar o teste de McNemar.")

Comparação estatística entre os dois melhores modelos por Macro F1
Modelo 1: SVM Linear - TF-IDF
Modelo 2: MLP - TF-IDF + SVD

b: casos em que o modelo 1 acertou e o modelo 2 errou = 97
c: casos em que o modelo 1 errou e o modelo 2 acertou = 84

Estatística de McNemar aproximada: 0.7956
p-valor: 0.3724

Conclusão: não há evidência estatística suficiente de diferença significativa entre os modelos.


In [489]:
# ============================================================
# Geração automática da submissão com base no melhor Macro F1
# ============================================================

import pandas as pd

CAMINHO_TESTE = "test.csv"
CAMINHO_SAMPLE = "/Users/annesakai/Downloads/PLN/template-implementacao/sample_submission.csv"
CAMINHO_SUBMISSION = "submission.csv"

test = pd.read_csv(CAMINHO_TESTE)
sample_submission = pd.read_csv(CAMINHO_SAMPLE)

# Se o treino usou texto pré-processado, aplica o mesmo pré-processamento ao teste
if TEXT_COL == "Body_preprocessado":
    test["Body_preprocessado"] = test["Body"].apply(
        lambda texto: preprocessar_texto(
            texto,
            remover_acentuacao=False,
            remover_nums=True,
            remover_stops=True,
            tamanho_minimo_token=3
        )
    )

TEXT_COL_TEST = TEXT_COL if TEXT_COL in test.columns else "Body"

X_sub = test[TEXT_COL_TEST].astype(str)

melhor_resultado = resultados_df.sort_values(
    "f1_macro",
    ascending=False
).iloc[0]

melhor_modelo_nome = melhor_resultado["modelo"]
melhor_representacao = melhor_resultado["representacao"]

print("Melhor modelo selecionado:")
print(melhor_modelo_nome, "-", melhor_representacao)

# ------------------------------------------------------------
# Recria o melhor modelo
# ------------------------------------------------------------

if melhor_modelo_nome == "Naive Bayes":
    from sklearn.naive_bayes import MultinomialNB
    modelo_final = MultinomialNB()

elif melhor_modelo_nome == "Regressão Logística":
    from sklearn.linear_model import LogisticRegression
    modelo_final = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )

elif melhor_modelo_nome == "SVM Linear":
    from sklearn.svm import LinearSVC
    modelo_final = LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )

elif melhor_modelo_nome == "MLP":
    from sklearn.neural_network import MLPClassifier
    modelo_final = MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        batch_size=64,
        learning_rate_init=0.001,
        max_iter=20,
        random_state=RANDOM_STATE,
        early_stopping=True
    )

else:
    raise ValueError(f"Modelo não reconhecido: {melhor_modelo_nome}")

# ------------------------------------------------------------
# Recria a representação textual com todos os dados de treino
# ------------------------------------------------------------

X_treino_completo = df_modelagem[TEXT_COL].astype(str)
y_treino_completo = df_modelagem[TARGET_COL].astype(int)

if melhor_representacao == "TF-IDF":
    X_treino_final = tfidf_vectorizer.fit_transform(X_treino_completo)
    X_sub_final = tfidf_vectorizer.transform(X_sub)

elif melhor_representacao == "Bag of Words":
    X_treino_final = bow_vectorizer.fit_transform(X_treino_completo)
    X_sub_final = bow_vectorizer.transform(X_sub)

elif melhor_representacao == "TF-IDF + SVD":
    X_treino_tfidf = tfidf_vectorizer.fit_transform(X_treino_completo)
    X_treino_svd = svd.fit_transform(X_treino_tfidf)
    X_treino_final = scaler.fit_transform(X_treino_svd)

    X_sub_tfidf = tfidf_vectorizer.transform(X_sub)
    X_sub_svd = svd.transform(X_sub_tfidf)
    X_sub_final = scaler.transform(X_sub_svd)

else:
    raise ValueError(f"Representação não reconhecida: {melhor_representacao}")

# ------------------------------------------------------------
# Treina no conjunto completo e gera predições para submissão
# ------------------------------------------------------------

modelo_final.fit(X_treino_final, y_treino_completo)

predicoes_submission = modelo_final.predict(X_sub_final)

# ------------------------------------------------------------
# Monta o arquivo de submissão
# ------------------------------------------------------------

submission = pd.DataFrame({
    "Id": test["Id"].values,
    "Category": predicoes_submission.astype(int)
})

submission.to_csv(CAMINHO_SUBMISSION, index=False)

display(submission.head())
print("Formato da submissão:", submission.shape)

print("\nDistribuição das classes previstas:")
print(submission["Category"].value_counts().sort_index())

print(f"\nArquivo salvo como: {CAMINHO_SUBMISSION}")

Melhor modelo selecionado:
SVM Linear - TF-IDF


,Id,Category
0,9213,0
1,17427,1
2,4753,3
3,4640,3
4,20412,3


Formato da submissão: (2521, 2)

Distribuição das classes previstas:
Category
0      84
1     376
2      52
3    1662
4     347
Name: count, dtype: int64

Arquivo salvo como: submission.csv
